# Documentación del Proceso ETL: Consolidación Histórica y Enriquecimiento Espacial del Valor de Referencia

## 1. Fuentes de Datos e Ingesta

El proyecto se alimenta de tres fuentes oficiales abiertas provistas por el Distrito Capital a través del portal de Datos Abiertos Bogotá e IDECA:

Valor de Referencia por Metro Cuadrado de Terreno: Conjunto de datos históricos en formato comprimido (.zip) que agrupa los polígonos de manzanas catastrales y sus respectivos valores económicos de referencia calculados año tras año (2012–2026).

Fuente: https://datosabiertos.bogota.gov.co/dataset/valor-de-referencia-por-metro-cuadrado-de-terreno

Sector Catastral: Capa geográfica oficial (.zip / Shapefile) que define la división político-administrativa a nivel de barrios o sectores catastrales, utilizada para contextualizar espacialmente cada manzana.

Fuente: https://datosabiertos.bogota.gov.co/dataset/sector-catastral

Localidad (loca.gpkg): Capa geoespacial en formato GeoPackage que delimita las localidades del Distrito Capital, permitiendo establecer la jerarquía territorial superior.

Fuente: https://datosabiertos.bogota.gov.co/dataset/localidad-bogota-d-c

## 2. Metodología y Procesamiento (ETL Pipeline)

El pipeline automatizado implementa un proceso de Extracción, Transformación y Carga (ETL) espacial estructurado en los siguientes pasos:

Ingestión y Estandarización: Se descomprimen de forma masiva los archivos históricos anuales, extrayendo las geometrías de las manzanas y normalizando los nombres de columnas heterogéneas (mancodigo, v_ref, shape_area, shape_len, ano) para unificarlas en un único marco de datos temporal.

Enriquecimiento Espacial por Intersección (sjoin): Utilizando la librería geopandas, se reproyectan las capas bajo un mismo sistema de referencia de coordenadas (CRS) y se ejecutan cruces espaciales secuenciales basados en el predicado intersects. Esto permite asignar a cada manzana histórica su respectivo barrio (desde el Sector Catastral) y localidad (desde el archivo GeoPackage).

Deduplicación e Integridad Estructural: Para corregir la multiplicación de registros derivada de manzanas que intersectan bordes administrativos, se implementa una limpieza basada en el identificador único original (id_original), garantizando una única asignación espacial por manzana.

Depuración de Datos Incompletos: Se aplica un filtro de exclusión (dropna) para remover registros que carezcan de variables económicas o administrativas esenciales (barrio, localidad, v_ref, shape_area, shape_len).

## 3. Productos Finales Generados

Como resultado del procesamiento, se obtienen dos datasets estructurados:

valor_referencia_historico_tabular.csv: Archivo tabular optimizado para análisis estadísticos y econométricos. Contiene el histórico consolidado a nivel de manzana, con sus valores de referencia, dimensiones físicas y jerarquía administrativa (barrio y localidad).

valor_referencia_historico_espacial.gpkg: Archivo GeoPackage que conserva la geometría vectorial completa, permitiendo visualizaciones geográficas y análisis de autocorrelación espacial en software SIG (QGIS, ArcGIS) o bibliotecas de Python.

In [33]:
import os
import zipfile
import glob
import re
import pandas as pd
import geopandas as gpd
from osgeo import gdal

gdal.SetConfigOption("SHAPE_RESTORE_SHX", "YES")

os.makedirs('data_historica', exist_ok=True)
for f in glob.glob('*.zip'):
    if 'sector' not in f.lower():
        with zipfile.ZipFile(f, 'r') as z:
            z.extractall(os.path.join('data_historica', os.path.splitext(f)[0]))

os.makedirs('temp_sector', exist_ok=True)
for f in glob.glob('*sector*.zip'):
    with zipfile.ZipFile(f, 'r') as z:
        z.extractall('temp_sector')

lista_gdfs = []
for shp in glob.glob('data_historica/**/*.shp', recursive=True):
    gdf = gpd.read_file(shp)
    match = re.search(r'20\d{2}', shp)
    ano = int(match.group(0)) if match else None
    cols = {c.lower(): c for c in gdf.columns}
    datos = {
        'mancodigo': gdf[cols.get('mancodigo')] if cols.get('mancodigo') else None,
        'v_ref': gdf[cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')] if (cols.get('v_ref') or cols.get('valor_refe') or cols.get('valref')) else None,
        'shape_area': gdf[cols.get('shape_area')] if cols.get('shape_area') else None,
        'shape_len': gdf[cols.get('shape_leng') or cols.get('shape_len')] if (cols.get('shape_leng') or cols.get('shape_len')) else None,
        'ano': ano,
        'geometry': gdf['geometry']
    }
    lista_gdfs.append(gpd.GeoDataFrame(datos, crs=gdf.crs))

gdf_maestro = pd.concat(lista_gdfs, ignore_index=True)
gdf_maestro = gpd.GeoDataFrame(gdf_maestro, geometry='geometry')
gdf_maestro = gdf_maestro.reset_index().rename(columns={'index': 'id_original'})
print(f"1. Registros en gdf_maestro inicial: {len(gdf_maestro)}")

archivo_sector = glob.glob('temp_sector/**/*.shp', recursive=True)[0]
gdf_sector = gpd.read_file(archivo_sector)
if gdf_maestro.crs != gdf_sector.crs:
    gdf_sector = gdf_sector.to_crs(gdf_maestro.crs)

gdf_sector_sub = gdf_sector[['SCANOMBRE', 'geometry']].rename(columns={'SCANOMBRE': 'barrio'})
gdf_enriquecido = gpd.sjoin(gdf_maestro, gdf_sector_sub, how='left', predicate='intersects')
if 'index_right' in gdf_enriquecido.columns:
    gdf_enriquecido = gdf_enriquecido.drop(columns=['index_right'])
print(f"2. Registros tras sjoin con barrio: {len(gdf_enriquecido)}")

gdf_localidad = gpd.read_file('loca.gpkg')
if gdf_enriquecido.crs != gdf_localidad.crs:
    gdf_localidad = gdf_localidad.to_crs(gdf_enriquecido.crs)

gdf_localidad_sub = gdf_localidad[['LocNombre', 'geometry']].rename(columns={'LocNombre': 'localidad'})
gdf_enriquecido = gpd.sjoin(gdf_enriquecido, gdf_localidad_sub, how='left', predicate='intersects')
if 'index_right' in gdf_enriquecido.columns:
    gdf_enriquecido = gdf_enriquecido.drop(columns=['index_right'])
print(f"3. Registros tras sjoin con localidad: {len(gdf_enriquecido)}")

gdf_enriquecido = gdf_enriquecido.drop_duplicates(subset=['id_original'], keep='first')
gdf_enriquecido = gdf_enriquecido.drop(columns=['id_original'])
print(f"4. Registros tras limpiar duplicados por ID original: {len(gdf_enriquecido)}")

gdf_enriquecido = gdf_enriquecido.dropna(subset=['barrio', 'localidad', 'v_ref', 'shape_area', 'shape_len'])
print(f"5. Registros finales tras eliminar nulos esenciales: {len(gdf_enriquecido)}")

pd.DataFrame(gdf_enriquecido).to_csv('valor_referencia_historico_tabular.csv', index=False)
gdf_enriquecido.to_file('valor_referencia_historico_espacial.gpkg', driver='GPKG')
print("¡Proceso completado y archivos exportados con éxito!")

1. Registros en gdf_maestro inicial: 477077
2. Registros tras sjoin con barrio: 516447
3. Registros tras sjoin con localidad: 522201
4. Registros tras limpiar duplicados por ID original: 477077
5. Registros finales tras eliminar nulos esenciales: 477029
¡Proceso completado y archivos exportados con éxito!


In [34]:
df = pd.read_csv('valor_referencia_historico_tabular.csv')
print("Localidades:", sorted(df['localidad'].dropna().unique()))
print(f"Total de localidades: {df['localidad'].nunique()}")
print(f"Total de barrios únicos: {df['barrio'].nunique()}")

/tmp/ipykernel_1510/3330608249.py:1: DtypeWarning: Columns (0) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('valor_referencia_historico_tabular.csv')


Localidades: ['ANTONIO NARIÑO', 'BARRIOS UNIDOS', 'BOSA', 'CANDELARIA', 'CHAPINERO', 'CIUDAD BOLIVAR', 'ENGATIVA', 'FONTIBON', 'KENNEDY', 'LOS MARTIRES', 'PUENTE ARANDA', 'RAFAEL URIBE URIBE', 'SAN CRISTOBAL', 'SANTA FE', 'SUBA', 'SUMAPAZ', 'TEUSAQUILLO', 'TUNJUELITO', 'USAQUEN', 'USME']
Total de localidades: 20
Total de barrios únicos: 1039


In [35]:
df.head(10)

,mancodigo,v_ref,shape_area,shape_len,ano,geometry,barrio,localidad
0,003201039,2200000.0,7.318068e-07,0.004886,2025,POLYGON ((-74.08379815899991 4.591266770000061...,SAN BERNARDO,SANTA FE
1,002432009,3150000.0,2.083963e-07,0.002359,2025,POLYGON ((-74.15059074699991 4.573440389000041...,ARBORIZADORA BAJA,CIUDAD BOLIVAR
2,005657092,720000.0,7.810987e-08,0.001136,2025,"POLYGON ((-74.1463936799999 4.711376722000068,...",SABANA DEL DORADO,ENGATIVA
3,006402005,2360000.0,2.819338e-08,0.000717,2025,POLYGON ((-74.13848724699994 4.684827686000062...,LA CABANA FONTIBON,FONTIBON
4,002521015,650000.0,8.719964e-08,0.001492,2025,POLYGON ((-74.14006067299994 4.559116009000093...,LUCERO ALTO,CIUDAD BOLIVAR
5,005637079,2400000.0,3.180436e-06,0.009100,2025,"POLYGON ((-74.1380452009999 4.707592588000068,...",AEROPUERTO EL DORADO,FONTIBON
6,006402060,2360000.0,3.336503e-07,0.002704,2025,POLYGON ((-74.13881981499992 4.684557087000087...,LA CABANA FONTIBON,FONTIBON
7,002521014,650000.0,5.799995e-08,0.001092,2025,POLYGON ((-74.14034279599991 4.558895254000049...,LUCERO ALTO,CIUDAD BOLIVAR
8,002414027,1040000.0,3.073642e-07,0.002589,2025,POLYGON ((-74.16948457199993 4.582224375000067...,ISMAEL PERDOMO,CIUDAD BOLIVAR
9,002515011,2300000.0,3.419954e-07,0.002615,2025,POLYGON ((-74.14776032299994 4.563349932000051...,SAN FRANCISCO,CIUDAD BOLIVAR
